In [1]:
import re
import hashlib
import unicodedata
import numpy as np
import pandas as pd

SEED = 42
RAW_PATH = "/kaggle/input/datasets/ravinarsimha/all-data/all_data.csv"
CLEAN_PATH = "/kaggle/working/english_reviews.csv"

In [2]:
df = pd.read_csv(RAW_PATH)

print(df.shape)
print(df.dtypes)
df.head()

(19985, 12)
Unnamed: 0           int64
Review_Language     object
City Name           object
Hotel Name          object
Upside_Review       object
Downside_Review     object
Review_Score       float64
Sentiment           object
source               int64
Prompt_Language     object
na_up_review          bool
na_down_review        bool
dtype: object


,Unnamed: 0,Review_Language,City Name,Hotel Name,Upside_Review,Downside_Review,Review_Score,Sentiment,source,Prompt_Language,na_up_review,na_down_review
0,0,Chinese,Ankara,迪万卡拉汉酒店,性价比超高的酒店，虽然价格比较贵，但是享受到的服务环境都还不错。,NaN,10.0,POS,0,NaN,False,True
1,1,Chinese,Ankara,迪万卡拉汉酒店,奥斯曼风格的建筑很有特色，房间装修精致，距离安卡拉城堡非常近，附近餐厅多，纪念品小商店多,不知道是不是因为夏季的原因，房间的空调不太给力，房间热；酒店员工会在窗外聊天，不考虑客人是不...,4.0,NEG,0,NaN,False,False
2,2,Chinese,Ankara,迪万卡拉汉酒店,非常喜歡酒店的環境氛圍，特別是通過地下通道到達博物館，這是一個驚喜（但入住時酒店服務人員并未...,酒店的空調令人傷心，第一天房間空調不起作用，次日上午出門時告知酒店經理，回來后依然沒有處理，...,5.0,NEG,0,NaN,False,False
3,3,Chinese,Ankara,安卡拉卡亚酒店,NaN,冷氣每隔5-15分鐘就會自動關閉，重新開啟有時候也要按三次才會開啟，飯店人員並沒有很積極地幫...,4.0,NEG,0,NaN,True,False
4,4,Chinese,Beijing,北京长城饭店(原北京喜来登长城饭店）,雖因為疫情影響加上老牌酒店旅客很少，但五星級的風骨猶存~這是寶藏好好傳承，比新酒店服務不到位強百倍,NaN,9.0,POS,0,NaN,False,True


# Review per language and class

In [3]:
print(df["Review_Language"].value_counts())

print(pd.crosstab(df["Review_Language"], df["source"], margins=True))

print(pd.crosstab(df["Review_Language"], df["Sentiment"]))

Review_Language
English     2000
French      2000
Romanian    2000
German      2000
Italian     2000
Korean      2000
Spanish     2000
Russian     2000
Turkish     2000
Chinese     1985
Name: count, dtype: int64
source              0      1    All
Review_Language                    
Chinese           985   1000   1985
English          1000   1000   2000
French           1000   1000   2000
German           1000   1000   2000
Italian          1000   1000   2000
Korean           1000   1000   2000
Romanian         1000   1000   2000
Russian          1000   1000   2000
Spanish          1000   1000   2000
Turkish          1000   1000   2000
All              9985  10000  19985
Sentiment         NEG   POS
Review_Language            
Chinese           985  1000
English          1000  1000
French           1000  1000
German           1000  1000
Italian          1000  1000
Korean           1000  1000
Romanian         1000  1000
Russian          1000  1000
Spanish          1000  1000
Turkish     

#  missing values and meta data columns.

In [4]:
print(df.isna().sum())

print(pd.crosstab(df["Prompt_Language"].isna(), df["source"]))

# Missingness flags differ by class
print(pd.crosstab(df["source"], [df["na_up_review"], df["na_down_review"]]))

Unnamed: 0            0
Review_Language       0
City Name             0
Hotel Name            0
Upside_Review       997
Downside_Review    2195
Review_Score          0
Sentiment             0
source                0
Prompt_Language    9985
na_up_review          0
na_down_review        0
dtype: int64
source              0      1
Prompt_Language             
False               0  10000
True             9985      0
na_up_review   False       True 
na_down_review False True  False
source                          
0               7386  1768   831
1               9413   422   165


In [5]:
eng_preview = df[df["Review_Language"] == "English"]
print(eng_preview.shape)
print(pd.crosstab(eng_preview["source"], eng_preview["Sentiment"]))
print(eng_preview["Prompt_Language"].value_counts())

(2000, 12)
Sentiment  NEG  POS
source             
0          500  500
1          500  500
Prompt_Language
English(Romanian)    66
Turkish              64
Korean               63
Russian              63
English(Russian)     62
German               61
English(Italian)     60
Spanish              60
English(French)      59
Romanian             54
Italian              53
English(German)      51
Chinese              50
English(Turkish)     50
English(Spanish)     48
English(Chinese)     48
English(Korean)      47
French               41
Name: count, dtype: int64


# keeping english

In [6]:
eng = df[df["Review_Language"] == "English"].copy()

eng = eng.reset_index().rename(columns={"index": "id"})
eng = eng[["id", "Upside_Review", "Downside_Review", "Hotel Name", "City Name", "source", "Sentiment"]]
eng = eng.rename(columns={"source": "label", "Sentiment": "sentiment"})

# sanity checks
print(eng.shape)
print(eng["label"].value_counts())
print(pd.crosstab(eng["label"], eng["sentiment"]))
print(eng["id"].is_unique)

(2000, 7)
label
0    1000
1    1000
Name: count, dtype: int64
sentiment  NEG  POS
label              
0          500  500
1          500  500
True


# building review text

In [7]:
eng["Upside_Review"]   = eng["Upside_Review"].fillna("")
eng["Downside_Review"] = eng["Downside_Review"].fillna("")
eng["text"] = (eng["Upside_Review"] + " " + eng["Downside_Review"]).str.strip()

eng["text"] = (eng["text"]
              .apply(lambda t: unicodedata.normalize("NFKC", t))
              .str.replace("’", "'", regex=False))

def mask_entities(row):
    t, name, city = row["text"], str(row["Hotel Name"]), str(row["City Name"])
    variants = {name,
                re.sub(r"^(the )?hotel ", "", name, flags=re.I),
                re.sub(r" hotel$", "", name, flags=re.I)}
    for v in sorted(variants, key=len, reverse=True):
        if len(v) > 3:
            t = re.sub(re.escape(v), " hotelname ", t, flags=re.I)
    t = re.sub(r"\b" + re.escape(city) + r"\b", " cityname ", t, flags=re.I)
    return re.sub(r"\s+", " ", t).strip()

eng["text"] = eng.apply(mask_entities, axis=1)

print(eng.groupby("label")["text"].apply(lambda s: s.str.contains(r"\bhotelname\b").mean()).round(3))
print(eng.groupby("label")["text"].apply(lambda s: s.str.contains(r"\bcityname\b").mean()).round(3))

eng = eng[["id", "text", "label", "sentiment"]]

label
0    0.004
1    0.269
Name: text, dtype: float64
label
0    0.077
1    0.275
Name: text, dtype: float64


# Data quality check

In [8]:
norm = eng["text"].str.lower().str.replace(r"[^a-z ]", "", regex=True).str.split().str.join(" ")
print("exact duplicates:", eng["text"].duplicated().sum())
print("duplicates after normalisation:", norm.duplicated().sum())
print(eng.loc[norm.duplicated(keep=False), ["id", "label", "text"]])

print("mojibake:", eng["text"].str.contains("â€|Ã").sum())
print("HTML entities:", eng["text"].str.contains(r"&[a-z]+;|&#").sum())

nonlat = eng["text"].apply(lambda t: sum(c.isalpha() and ord(c) > 591 for c in t)
                                    / max(1, sum(c.isalpha() for c in t)))
print("reviews with >10% non-Latin letters:", (nonlat > 0.1).sum())

exact duplicates: 0
duplicates after normalisation: 1
       id  label             text
274  1259      0  Great location!
893  1878      0  Great location.
mojibake: 0
HTML entities: 0
reviews with >10% non-Latin letters: 0


# Sanity check for text and review lengths

In [9]:
print((eng["text"] == "").sum())
print(eng["text"].str.contains(r"\bnan\b").sum())
print(eng["text"].str.split().str.len().describe())
print(eng.groupby("label")["text"].apply(lambda s: s.str.split().str.len().mean()))
print(eng.sample(3, random_state=1)["text"].tolist())

0
0
count    2000.00000
mean       51.76450
std        43.10637
min         2.00000
25%        22.75000
50%        43.00000
75%        70.00000
max       548.00000
Name: text, dtype: float64
label
0    48.608
1    54.921
Name: text, dtype: float64
['friendliness of staff', "The location of hotelname was convenient for exploring the city. However, our experience at the hotel was below par. The room was far smaller than expected from the photographs online and lacked certain basic amenities. We found the bed uncomfortable and the cleanliness in some areas questionable. The noise from the adjacent rooms was audible all night. We also found the staff somewhat indifferent and unhelpful during many occasions throughout our stay. We overall felt like we didn't get the value for our money.", "The hotel's conference centre and location are outstanding. Some rooms need a bit of a refurbishment."]


# stratified split

In [10]:
from sklearn.model_selection import train_test_split

strat_key = eng["label"].astype(str) + "_" + eng["sentiment"]

train_df, test_df = train_test_split(
    eng,
    test_size=0.25,
    stratify=strat_key,
    random_state=SEED,
)

# checksum and saving

In [11]:
eng["split"] = "train"
eng.loc[test_df.index, "split"] = "test"

eng.to_csv(CLEAN_PATH, index=False)
print("data checksum (MD5):", hashlib.md5(open(CLEAN_PATH, "rb").read()).hexdigest())

data checksum (MD5): 5cda9feaf803a53f03316c9b823cfc1e


# split checking 

In [12]:
print(eng["split"].value_counts())
print(pd.crosstab([eng["split"], eng["label"]], eng["sentiment"]))
print(set(train_df["id"]) & set(test_df["id"]))

check = pd.read_csv(CLEAN_PATH)
print(check.shape, check.columns.tolist())

split
train    1500
test      500
Name: count, dtype: int64
sentiment    NEG  POS
split label          
test  0      125  125
      1      125  125
train 0      375  375
      1      375  375
set()
(2000, 5) ['id', 'text', 'label', 'sentiment', 'split']


Vectorizer

In [13]:
from sklearn.feature_extraction.text import CountVectorizer

TOKEN_PATTERN = r"(?u)\b[^\W\d_]{2,}\b"

def make_vectorizer(min_df=0.01):
    return CountVectorizer(
        lowercase=True,
        token_pattern=TOKEN_PATTERN,
        stop_words=None,
        ngram_range=(1, 2),
        min_df=min_df
    )

In [14]:
data  = pd.read_csv(CLEAN_PATH)
train = data[data["split"] == "train"]
test  = data[data["split"] == "test"]

vec = make_vectorizer()
X_train = vec.fit_transform(train["text"])
X_test  = vec.transform(test["text"])
y_train, y_test = train["label"].values, test["label"].values

vocab = vec.get_feature_names_out()
pd.Series(vocab, name="term").to_csv("/kaggle/working/vocabulary.csv", index=False)

# vocab and sparsity threshold check.

In [15]:
print(X_train.shape, X_test.shape)
print(sum(" " in t for t in vocab), "bigrams")
print(vocab[:15])

print((X_test.sum(axis=1) == 0).sum(), "test reviews with no known terms")

for md in [1, 0.005, 0.01, 0.02, 0.05]:
    v = CountVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN,
                        ngram_range=(1, 2), min_df=md).fit(train["text"])
    print(md, len(v.vocabulary_))

(1500, 1133) (500, 1133)
527 bigrams
['about' 'about the' 'above' 'absolutely' 'ac' 'access' 'access to'
 'accessible' 'accommodating' 'additionally' 'additionally the' 'after'
 'again' 'air' 'air conditioning']
0 test reviews with no known terms
1 35389
0.005 2103
0.01 1133
0.02 547
0.05 183
